# About

Project Goal:
Train a YOLO object detection model to identify:

- Workers **with hard hats**
- Workers **without hard hats**
- **head location**

This simulates a safety inspection use case in industrial environments.

Dataset: https://www.kaggle.com/datasets/andrewmvd/hard-hat-detection

**About the Dataset:**

Improve workplace safety by detecting people and hard hats on 5k images with bbox annotations.

This dataset, contains 5000 images with bounding box annotations in the PASCAL VOC format for these 3 classes:

Helmet;

Person;

Head.

# Load and general setup

## Imports


In [ ]:
import os
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import xml.etree.ElementTree as ET #used to read and parse XML files
import numpy as np
import random


## Import model + define some general functions

In [ ]:
!pip install ultralytics

In [ ]:
#Load the model
from ultralytics import YOLO

# Load a COCO-pretrained YOLOv8n model
model = YOLO("yolov8n.pt")

# Display model information (optional)
model.info()

In [ ]:
#Function to display the results/prediction
def display_results(results):
  # print results: get the first result of 1st image (it is one image anyways here but still we have to get it)
  result = results[0]

  # get boxes
  boxes = result.boxes.xyxy  # x1, y1, x2, y2
  scores = result.boxes.conf  # confidence scores
  classes = result.boxes.cls  # class indices

  # Get class names (of the objects detected)
  class_names = [result.names[int(c)] for c in classes]

  # Print
  for i, box in enumerate(boxes):
      x1, y1, x2, y2 = box
      print(f"Object: {class_names[i]}, Confidence: {scores[i]:.2f}, Box: [{x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}]")

  # Show an image in bounding boxes
  result.show()



## Download Dataset

## General

In [ ]:
dataset_path = "/content/Dataset/1"
images_path = os.path.join(dataset_path, 'images')
annotations_path = os.path.join(dataset_path, 'annotations')

# EDA

## Verifying labels in dataset

In [ ]:
# Let's check what labels are ACTUALLY in your dataset
import xml.etree.ElementTree as ET

def check_dataset_labels():
    """
    Check all unique labels in the dataset
    """
    all_labels = set()

    annotation_files = os.listdir(annotations_path)

    for annot_file in annotation_files[:20]:  # Check first 20 files
        annot_path = os.path.join(annotations_path, annot_file)
        tree = ET.parse(annot_path)
        root = tree.getroot()

        # Get all labels
        for obj in root.findall('object'):
            label = obj.find('name').text
            all_labels.add(label)

    print("Unique labels found in dataset:")
    print(all_labels)

    return all_labels

# Run this
check_dataset_labels()

In [ ]:
#Function to find images having certain labels
def find_images_with_label(target_label):
    """
    Find which images contain a specific label
    """
    images_with_label = []

    annotation_files = os.listdir(annotations_path)

    for annot_file in annotation_files:
        annot_path = os.path.join(annotations_path, annot_file)
        tree = ET.parse(annot_path)
        root = tree.getroot()

        # Check if this annotation has the target label
        labels_in_image = [obj.find('name').text for obj in root.findall('object')]

        if target_label in labels_in_image:
            image_name = root.find('filename').text
            images_with_label.append((image_name, labels_in_image))

    print(f"Found {len(images_with_label)} images with '{target_label}' label")
    print(f"\nFirst 5 examples:")
    for img, labels in images_with_label[:5]:
        print(f"  {img}: {labels}")

    return images_with_label

# Check for 'person' labels
person_images = find_images_with_label('person')

## Visualize images

In [ ]:
def visualize_one_image(image_name):
    """
    Visualize a single image with its bounding boxes

    How it works:
    1. Read the image
    2. Read the annotation XML file
    3. Draw boxes on the image
    4. Display it
    """
    #Step 1: Read the image
    # Build full path to image
    image_path = os.path.join(images_path, image_name)
    # Read image using OpenCV (reads as BGR)
    image = cv2.imread(image_path)
    # Convert BGR to RGB (for matplotlib display)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    #Step 2: Read the annotation XML file
    # Get annotation filename (same name but .xml extension)
    annotation_name = image_name.replace('.png', '.xml').replace('.jpg', '.xml')
    annotation_path = os.path.join(annotations_path, annotation_name)
    # Parse XML
    tree = ET.parse(annotation_path)
    root = tree.getroot()

    # STEP 3: Draw boxes on image
    # Create figure and axis
    fig, ax = plt.subplots(1, figsize=(12,8))
    ax.imshow(image)
    # Loop through all objects in annotation
    for obj in root.findall('object'):
        # Get label (e.g., "helmet", "head", "person")
        label = obj.find('name').text

        # Get bounding box coordinates
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)

        # Calculate width and height
        width = xmax - xmin
        height = ymax - ymin

        # Create rectangle patch
        rect = patches.Rectangle(
            (xmin, ymin), width, height,
            linewidth=2, edgecolor='red', facecolor='none'
        )

        # Add rectangle to plot
        ax.add_patch(rect)

        # Add label text
        ax.text(xmin, ymin-5, label,
                bbox=dict(facecolor='red', alpha=0.5),
                fontsize=10, color='white')

    # STEP 4: Display
    ax.axis('off')
    plt.title(f"Image: {image_name}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Get an image and visualize it
image_files = os.listdir(images_path)
print(f"Found {len(image_files)} images")
print(f"Testing with: {image_files[0]}")

# Visualize it
visualize_one_image(image_files[0])

#Visualize image with person label
image_with_person_name = 'hard_hat_workers1007.png'
visualize_one_image(image_with_person_name)

## Class Distribution Analysis

In [ ]:
def analyze_class_distribution():
    """
    Analyze how balanced/imbalanced your classes are
    """
    import matplotlib.pyplot as plt

    label_counts = {}

    for annot_file in os.listdir(annotations_path):
        annot_path = os.path.join(annotations_path, annot_file)
        tree = ET.parse(annot_path)
        root = tree.getroot()

        for obj in root.findall('object'):
            label = obj.find('name').text
            label_counts[label] = label_counts.get(label, 0) + 1

    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(label_counts.keys(), label_counts.values(), color=['red', 'blue', 'green'])
    plt.title('Class Distribution in Dataset', fontsize=14, fontweight='bold')
    plt.xlabel('Class')
    plt.ylabel('Number of Instances')
    plt.grid(axis='y', alpha=0.3)

    for label, count in label_counts.items():
        percentage = (count / sum(label_counts.values())) * 100
        print(f"{label}: {count} instances ({percentage:.1f}%)")

    plt.show()

    return label_counts

analyze_class_distribution()

# Dataset Preparation

The dataset is very imbalanced:

'helmet': 18966

'head': 5785

'person': 751

Person class will be removed, and it will be assumed that there is a helmet/head, then there is a person.


## Remove Person Annotation (from Original Dataset)

In [ ]:
def remove_person_class(annotations_path):
    """
    Remove all 'person' class labels from XML annotations
    Keeps only 'helmet' and 'head' classes

    Args:
    - annotations_path (str): Path to the directory containing XML annotations
    """
    total_files = 0
    modified_files = 0
    total_person_removed=0

    for annot_file in os.listdir(annotations_path): #os.listdir(annotations_path) returns a list of all files and folders inside annotations_path.
      if not annot_file.endswith('.xml'):
            continue

      annot_path = os.path.join(annotations_path, annot_file)
      total_files += 1
      # Parse XML
      tree = ET.parse(annot_path)
      root = tree.getroot()

      person_count = 0
      # Find and remove all 'person' objects
      for obj in root.findall('object'):
        if obj.find('name').text == 'person':
          root.remove(obj)
          person_count += 1

      if person_count>0:
        # Save modified XML
        tree.write(annot_path)
        modified_files += 1
        total_person_removed += person_count

    print(f"Summary:")
    print(f"- Total annotation files processed: {total_files}")
    print(f"- Files modified: {modified_files}")
    print(f"- Total 'person' instances removed: {total_person_removed}")
    print(f"- Remaining classes: helmet, head")



In [ ]:
annotations_path = os.path.join(dataset_path, 'annotations')
remove_person_class(annotations_path)

## Convert annotations to YOLO format (Do it once, already done, redo if issues)

**What happens step by step:**

1. **convert_xml_to_txt()** starts
   - Creates output folder if it doesn't exist already
   - Loops through all XML files
   - For each XML file:
     - Reads it with `ET.parse()`
     - Gets image name and size
     - Gets all bounding boxes (xmin, ymin, xmax, ymax)
     - Calls **save_txt_file()** ↓

2. **save_txt_file()** runs
   - Creates a .txt file
   - For each box:
     - Calls **convert_annot()** ↓
     - Writes the result to .txt file

3. **convert_annot()** runs
   - Takes pixel coordinates (xmin, ymin, xmax, ymax)
   - Converts to YOLO format (center_x, center_y, width, height)
   - Normalizes to 0-1 range
   - Returns the converted box

**Flow:**
```
convert_xml_to_txt()
  → reads XML
  → calls save_txt_file()
      → calls convert_annot() for each box
      → writes to .txt file

In [ ]:
#Class mapping
classes = ['helmet', 'head']

def convert_annot(size, box):
    """
    Convert Pascal VOC bbox to YOLO format

    Args:
        size: (width, height) of image
        box: [xmin, ymin, xmax, ymax] in pixels

    Returns:
        [center_x, center_y, width, height] normalized to 0-1
    """
    x1 = int(box[0])
    y1 = int(box[1])
    x2 = int(box[2])
    y2 = int(box[3])

    dw = np.float32(1. / int(size[0]))
    dh = np.float32(1. / int(size[1]))

    w = x2 - x1
    h = y2 - y1
    x = x1 + (w / 2)
    y = y1 + (h / 2)

    x = x * dw
    w = w * dw
    y = y * dh
    h = h * dh
    return [x, y, w, h]

def save_txt_file(save_path, img_name, size, boxes):
    """
    Save YOLO format annotations to txt file

    Args:
        save_path: Directory to save txt files
        img_name: Image filename (without extension)
        size: (width, height) of image
        boxes: List of [class_name, xmin, ymin, xmax, ymax]
    """
    txt_file = os.path.join(save_path, img_name + '.txt')

    with open(txt_file, 'w') as f:
        for box in boxes:
            cls_num = classes.index(box[0])
            yolo_box = convert_annot(size, box[1:])
            f.write(f"{cls_num} {yolo_box[0]:.6f} {yolo_box[1]:.6f} {yolo_box[2]:.6f} {yolo_box[3]:.6f}\n")

def convert_xml_to_txt(annotations_folder, output_folder):
    """
    Read all XML files, extract boxes, save as TXT in YOLO format

    Args:
        annotations_folder: Folder containing XML files
        output_folder: Folder to save TXT files
    """
    # Create output folder if it doesn't exist already
    os.makedirs(output_folder, exist_ok=True)

    converted_count = 0

    # Loop through all XML files
    for xml_file in os.listdir(annotations_folder):
        if not xml_file.endswith('.xml'):
            continue

        xml_path = os.path.join(annotations_folder, xml_file)
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Get image info
        img_filename = root.find('filename').text
        img_name = os.path.splitext(img_filename)[0]  # Remove extension

        size_elem = root.find('size')
        img_width = int(size_elem.find('width').text)
        img_height = int(size_elem.find('height').text)
        size = (img_width, img_height)

        # Get all boxes
        boxes = []
        for obj in root.findall('object'):
            class_name = obj.find('name').text

            # Skip if not in our classes
            if class_name not in classes:
                continue

            bbox = obj.find('bndbox')
            xmin = int(bbox.find('xmin').text)
            ymin = int(bbox.find('ymin').text)
            xmax = int(bbox.find('xmax').text)
            ymax = int(bbox.find('ymax').text)

            boxes.append([class_name, xmin, ymin, xmax, ymax])

        # Save as TXT
        if len(boxes) > 0:  # Only save if there are valid boxes
            save_txt_file(output_folder, img_name, size, boxes)
            converted_count += 1

    print(f"Converted {converted_count} XML files to TXT format")
    print(f"Saved to: {output_folder}")
    return converted_count

In [ ]:
# Convert original annotations only
convert_xml_to_txt(
    annotations_folder=os.path.join(dataset_path, 'annotations'),
    output_folder=os.path.join(dataset_path, 'labels')
)

## Path to labels

In [ ]:
#Path to labels
labels_path = os.path.join(dataset_path, 'labels')

## Train Test Split and Save in folders

This creates directories for train, test and val data in the specified location.

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

# NEW: Path for split dataset
split_dataset_path = '/content/Dataset/2'

# Get all images (from annotations to ensure they have labels)
annotation_files = [f for f in os.listdir(annotations_path) if f.endswith('.xml')]

image_list = []
for annot_file in annotation_files:
    img_name = annot_file.replace('.xml', '')
    # Check for common extensions
    for ext in ['.png', '.jpg', '.jpeg']:
        if os.path.exists(os.path.join(images_path, img_name + ext)):
            image_list.append(img_name + ext)
            break

print(f"Total images: {len(image_list)}")

# Split 1: 70% train, 30% temp (which will become val+test)
train_list, temp_list = train_test_split(image_list, test_size=0.3, random_state=42)

# Split 2: Split temp into 50% val, 50% test (15% and 15% of total)
val_list, test_list = train_test_split(temp_list, test_size=0.5, random_state=42)

print(f"\nSplit:")
print(f"  Train: {len(train_list)} ({len(train_list)/len(image_list)*100:.1f}%)")
print(f"  Val:   {len(val_list)} ({len(val_list)/len(image_list)*100:.1f}%)")
print(f"  Test:  {len(test_list)} ({len(test_list)/len(image_list)*100:.1f}%)")

# Create folder structure
splits = {
    'train': train_list,
    'val': val_list,
    'test': test_list
}

for split_name, file_list in splits.items():
    # Create directories in the NEW location
    split_images_dir = os.path.join(split_dataset_path, split_name, 'images')
    split_labels_dir = os.path.join(split_dataset_path, split_name, 'labels')

    os.makedirs(split_images_dir, exist_ok=True)
    os.makedirs(split_labels_dir, exist_ok=True)

    print(f"\nCopying {split_name} files...")

    # Copy files
    for img_file in file_list:
        # Copy image
        src_img = os.path.join(images_path, img_file)
        dst_img = os.path.join(split_images_dir, img_file)
        shutil.copy(src_img, dst_img)

        # Copy label
        label_file = os.path.splitext(img_file)[0] + '.txt'
        src_label = os.path.join(labels_path, label_file)
        dst_label = os.path.join(split_labels_dir, label_file)
        shutil.copy(src_label, dst_label)


print("\nDataset split complete!\n")
print("="*60)
print(f"\nFolder structure:")
print(f"  {dataset_path}/train/images/  ({len(train_list)} images)")
print(f"  {dataset_path}/train/labels/  ({len(train_list)} labels)")
print(f"  {dataset_path}/val/images/    ({len(val_list)} images)")
print(f"  {dataset_path}/val/labels/    ({len(val_list)} labels)")
print(f"  {dataset_path}/test/images/   ({len(test_list)} images)")
print(f"  {dataset_path}/test/labels/   ({len(test_list)} labels)")

## YAML file (Run it once)

Why is YAML file needed:

When you train YOLO:
pythonmodel.train(data='path/to/data.yaml', epochs=50)

**YOLO reads the yaml file to know:**

- Where is the data? → path: /Dataset/2

- Where are train images? → train/images

- Where are val images? → val/images

- How many classes? → nc: 2

- What are the class names? → ['helmet', 'head']

That's it. The yaml file is just a config file that tells YOLO where everything is and what classes to detect.

In [ ]:
# Create data.yaml file
yaml_content = f"""path: {split_dataset_path}
train: train/images
val: val/images
test: test/images

nc: 2
names: ['helmet', 'head']
"""

yaml_path = os.path.join(split_dataset_path, 'data.yaml')

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"data.yaml created at: {yaml_path}")

# Train

## Paths to train, test, val (Path to yaml file)

In [ ]:
split_dataset_path = '/content/Dataset/2'
data_yaml_path = os.path.join(split_dataset_path, 'data.yaml')

## Train the model

Monitor Mertics:

Step 3: Monitor metrics (happens automatically during training)

YOLO will print: mAP, precision, recall after each epoch

Results saved to: runs/detect/helmet_detection/

In [ ]:
#note: had to change epoch, imgsz, and batch even though they are not the best but to be able to run
results = model.train(
    data=data_yaml_path,
    epochs=30,
    imgsz=640,
    batch=50,
    name='helmet_detection',
    patience=10, #for early stopping
    plots=True #Generate training plots automatically (loss curves, confusion matrix, PR curves)
)



## View Plots after training

In [ ]:
from IPython.display import Image, display

def view_plots(image_path: str, title="Results:", width=600):
    print(title, "\n")
    display(Image(filename=image_path, width=width))
    print()

In [ ]:
# View results
view_plots('/content/runs/detect/helmet_detection2/confusion_matrix.png', "Training: Confusion Matrix")
view_plots('/content/runs/detect/helmet_detection2/confusion_matrix_normalized.png', "Training: Confusion Matrix Normalized")
view_plots('/content/runs/detect/helmet_detection2/results.png', "Training: Results: ", width= 1000)
view_plots('/content/runs/detect/helmet_detection2/BoxPR_curve.png', "Training: PR Curves")


# Evaluation - Testing

In [ ]:
# Evaluate on test set
metrics = model.val(data=data_yaml_path, split='test')

# Print metrics
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

## Visualize predictions with bounding boxes

In [ ]:
#Prediction for 1 image
#First 5 test images paths
test_images = [
    os.path.join(split_dataset_path, 'test/images/hard_hat_workers1001.png'),
    os.path.join(split_dataset_path, 'test/images/hard_hat_workers1008.png'),
    os.path.join(split_dataset_path, 'test/images/hard_hat_workers102.png'),
    os.path.join(split_dataset_path, 'test/images/hard_hat_workers1029.png'),
    os.path.join(split_dataset_path, 'test/images/hard_hat_workers1032.png')
]

actual_labels= [
    os.path.join(split_dataset_path, 'test/labels/hard_hat_workers1001.txt'),
    os.path.join(split_dataset_path, 'test/labels/hard_hat_workers1008.txt'),
    os.path.join(split_dataset_path, 'test/labels/hard_hat_workers102.txt'),
    os.path.join(split_dataset_path, 'test/labels/hard_hat_workers1029.txt'),
    os.path.join(split_dataset_path, 'test/labels/hard_hat_workers1032.txt')
]

results = []
for path in test_images:
    results.append(model.predict(path, conf=0.5, show=False))  # Add show=False

# Visualize
for result in results:
    img_with_boxes = result[0].plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

# Actual and pred boxes

In [ ]:
def visualize_pred_vs_actual(image_path, label_path, model, conf=0.5):
    print("Green bozxes = Actual Labels\n", "Red boxes = Predictions\n")
    print("-"*60)
    # Read image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_h, img_w = image.shape[:2]

    # Predict
    results = model.predict(image_path, conf=conf, show=False)

    # Read actual labels
    actual_boxes = []
    with open(label_path, 'r') as f:
        for line in f:
            cls, cx, cy, w, h = map(float, line.strip().split())
            xmin = int((cx - w/2) * img_w)
            ymin = int((cy - h/2) * img_h)
            xmax = int((cx + w/2) * img_w)
            ymax = int((cy + h/2) * img_h)
            actual_boxes.append([int(cls), xmin, ymin, xmax, ymax])

    # Plot
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(image)

    # Green = actual labels
    for cls, x1, y1, x2, y2 in actual_boxes:
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                 edgecolor='green', facecolor='none', linestyle='--')
        ax.add_patch(rect)

    # Red = predictions
    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                 edgecolor='red', facecolor='none')
        ax.add_patch(rect)

    ax.axis('off')
    plt.title("Green=GT, Red=Pred")
    plt.show()

# Use
for img, label in zip(test_images, actual_labels):
    visualize_pred_vs_actual(img, label, model)


# Export the model

In [ ]:
# Load your trained model
model = YOLO('/content/runs/detect/helmet_detection2/weights/best.pt')

# Export to ONNX format (most common)
model.export(format='onnx')

# Model exported to: runs/detect/helmet_detection/weights/best.onnx

In [ ]:
# Download to your computer
from google.colab import files

files.download("/content/runs/detect/helmet_detection2/weights/best.onnx")